# <font color='056938'> **RETO de aplicación: Extracción de datos vía API** </font>


**Página web seleccionada:** Tiendas Branchos — https://www.tiendasbranchos.com/

**API utilizada:** API pública de catálogo de **VTEX** (la plataforma de e-commerce sobre la que está construida la tienda). Esta API no requiere llave ni autenticación: es la misma que usa el propio sitio web para mostrar los productos en pantalla.

Endpoint base:
```
https://www.tiendasbranchos.com/api/catalog_system/pub/products/search
```

## <font color='46B8A9'> **Importar librerías** </font>

In [ ]:
# Librerias necesarias
import requests as rq   # para hacer las peticiones (requests) a la API
import pandas as pd     # para construir el DataFrame final
import time              # para hacer pequenas pausas entre peticiones


## <font color='46B8A9'> **Paso 1: Armar la consulta** </font>

Toda tienda construida sobre VTEX expone automáticamente este endpoint público, sin necesidad de llave (API key) ni login. Es la misma URL que el navegador de la tienda consulta por detrás cuando tú buscas un producto; nosotros simplemente le hablamos directamente.

Para consultarla, le mandamos una petición HTTP tipo `GET` con parámetros en la url (*query parameters*). Los más importantes son:
* `ft` (*full text*): el texto de búsqueda, por ejemplo `ft=tenis`.
* `_from` y `_to`: el rango de productos que queremos. Esto es la **paginación**: la API nunca entrega "todo el catálogo" de una sola vez, solo el rango solicitado (máximo ~50 por petición). Por eso más adelante habrá un bucle que va moviendo ese rango hasta juntar al menos 30 productos.

Hagamos una primera consulta pequeña (pedimos solo 5 productos) para ver cómo luce la respuesta antes de automatizar todo.

In [ ]:

url_api = "https://www.tiendasbranchos.com/api/catalog_system/pub/products/search"

# Parametros de la consulta: buscamos "tenis" y pedimos los primeros 5 resultados
parametros = {
    "ft": "tenis",
    "_from": 0,
    "_to": 4
}
# Hacemos la peticion GET a la API
respuesta = rq.get(url_api, params=parametros)

# Convertimos la respuesta a una lista de diccionarios de Python
productos_json = respuesta.json()

print("Codigo de respuesta:", respuesta.status_code)
print("Numero de productos obtenidos:", len(productos_json))


## <font color='46B8A9'> **Paso 2: qué responde la API** </font>

A diferencia del HTML (que tocaba leer con `BeautifulSoup` buscando etiquetas `<h3>`, `<a>`, `<li>`), VTEX devuelve texto en formato **JSON**, que `requests` convierte directo en listas y diccionarios de Python con `.json()`. Cada producto llega como un diccionario con llaves ya nombradas, así que no hay que "buscar" nada con selectores raros: solo se accede a la llave.

In [ ]:
# Miremos como luce el primer producto de la lista
primer_producto = productos_json[0]
primer_producto


## <font color='46B8A9'> **Paso 3: extraer nombre, url, precio y tallas de un producto** </font>

Como todo el detalle ya viene en esa misma respuesta, extraer los datos es solo *navegar el diccionario*, sin necesidad de un segundo viaje a la pagina del producto. Dentro de cada producto, la informacion que necesitamos esta en estas llaves:

* `productName` -> nombre del producto
* `link` -> url completa del producto (la API ya la entrega completa, no toca armarla como con el `href`)
* `brand` -> marca
* `items` -> lista de variantes (tallas/colores). Cada item tiene `name` (la talla) y dentro de `sellers -> commertialOffer -> Price` el precio.

In [ ]:
# Extraemos el nombre del primer producto
nombre = primer_producto["productName"]
nombre


In [ ]:
# Extraemos el link
url_producto = primer_producto["link"]
url_producto


In [ ]:
# El precio esta "anidado" dentro de items -> sellers -> commertialOffer -> Price
# Tomamos el precio de la primera variante (items[0])
precio = primer_producto["items"][0]["sellers"][0]["commertialOffer"]["Price"]
precio


`items` es una lista. Aqui cada elemento de la lista es una talla/variante disponible del producto.

In [ ]:
# Agregue las tallas/variantes del producto
tallas = [item["name"] for item in primer_producto["items"]]
tallas


## <font color='46B8A9'> **Paso 4: iteramos sobre todos los productos (paginación)** </font>

La API entrega los productos en "páginas": no podemos pedir 1000 de una sola vez. Para conseguir por lo menos 30 productos, pedimos por bloques de 20, moviendo el rango `_from` / `_to` en cada vuelta, hasta completar lo que necesitamos.

Aquí está la ventaja clave frente al scraping: en el notebook de las recetas se necesitaban **dos rondas** de peticiones (una para la lista, otra por cada receta para sacar los ingredientes). Con la API, **cada bloque que pedimos ya trae todo el detalle** de cada producto (nombre, url, precio, tallas) — no hace falta un segundo viaje por producto.

In [ ]:
productos = []       # aqui iremos guardando [nombre, url, precio, marca, tallas] de cada producto
tamano_bloque = 20    # cuantos productos pedimos por cada peticion
minimo_deseado = 50   # cuantos productos queremos en total, como minimo

inicio = 0
while len(productos) < minimo_deseado:
  fin = inicio + tamano_bloque - 1

  parametros = {"ft": "tenis", "_from": inicio, "_to": fin}
  respuesta = rq.get(url_api, params=parametros)
  bloque = respuesta.json()

  # si la API ya no devuelve mas productos, paramos
  if len(bloque) == 0:
    break

  for producto in bloque:
    nombre = producto["productName"]
    url_producto = producto["link"]
    marca = producto.get("brand", "Sin marca")
    precio = producto["items"][0]["sellers"][0]["commertialOffer"]["Price"]
    tallas = [item["name"] for item in producto["items"]]

    productos.append([nombre, url_producto, precio, marca, tallas])

  inicio += tamano_bloque
  time.sleep(0.5)  # pequena pausa para no saturar la API

len(productos)


## <font color='46B8A9'> **Construir el DataFrame final** </font>

In [ ]:
df = pd.DataFrame(productos, columns=["nombre", "url", "precio", "marca", "tallas"])
df.head(10)


In [ ]:
# Confirmamos que tenemos al menos 30 productos
print("Total de productos obtenidos:", len(df))
